### 从老版本的chain迁移到LCEL
---
* 示例1：从LLMChain迁移
* 示例2：从StuffDocumentsChain迁移

#### 示例1：从LLMChain迁移

废弃的用法

In [ ]:
from langchain_classic.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="Qwen/Qwen3.8-27B",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

prompt = ChatPromptTemplate.from_messages(
    [("user", "Tell me a {adjective} joke")]
)

legacy_chain = LLMChain(llm=model, prompt=prompt)

legency_result = legacy_chain({"adjective": "funny"})

legency_result

C:\Users\Lxb\AppData\Local\Temp\ipykernel_63324\505858723.py:21: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  legacy_chain = LLMChain(llm=model, prompt=prompt)
C:\Users\Lxb\AppData\Local\Temp\ipykernel_63324\505858723.py:23: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  legency_result = legacy_chain({"adjective": "funny"})


{'adjective': 'funny',
 'text': '\n\nWhy did the scarecrow win an award?\n\nBecause he was outstanding in his field!'}

#### LCEL的用法

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="Qwen/Qwen3.8-27B",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

prompt = ChatPromptTemplate.from_messages(
    [("user", "Tell me a {adjective} joke")]
)

chain = prompt  | model | StrOutputParser()

chain.invoke({"adjective": "funny"})

'\n\nWhy did the scarecrow win an award?\n\nBecause he was outstanding in his field!'

#### 示例2：从StuffDocumentsChain迁移

废弃的用法

In [ ]:
from langchain_core.documents import Document

import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="Qwen/Qwen3.8-27B",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

documents = [
    Document(page_content="Apples are red", metadata={"title": "apple_book"}),
    Document(page_content="Blueberries are blue", metadata={"title": "blueberry_book"}),
    Document(page_content="Bananas are yellow", metadata={"title": "banana_book"}),
]

from langchain_classic.chains import LLMChain, StuffDocumentsChain
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# This controls how each document will be formatted. Specifically,
# it will be passed to `format_document` - see that function for more details.
document_prompt = PromptTemplate(
    input_variables=["page_content"], template="{page_content}"
)
document_variable_name = "context"

# The prompt here should take as an input variable the `document_variable_name`
prompt = ChatPromptTemplate.from_template("Summarize this content: {context}")
model_chain = LLMChain(llm=model, prompt=prompt)
chain = StuffDocumentsChain(
    llm_chain=model_chain,
    document_prompt=document_prompt,
    document_variable_name=document_variable_name
)

C:\Users\Lxb\AppData\Local\Temp\ipykernel_63324\3489709820.py:34: LangChainDeprecationWarning: The class `StuffDocumentsChain` was deprecated in LangChain 0.2.13 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build new RAG flows with `create_agent` and a retrieval tool. See https://docs.langchain.com/oss/python/langchain/rag
  chain = StuffDocumentsChain(


In [6]:
result = chain.invoke(documents)
result["output_text"]

'\n\nThe content lists common fruit colors: apples are red, blueberries are blue, and bananas are yellow.'

#### LCEL的用法

In [9]:
from langchain_core.documents import Document

import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="Qwen/Qwen3.8-27B",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

documents = [
    Document(page_content="Apples are red", metadata={"title": "apple_book"}),
    Document(page_content="Blueberries are blue", metadata={"title": "blueberry_book"}),
    Document(page_content="Bananas are yellow", metadata={"title": "banana_book"}),
]

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Summarize this content: {context}")
chain = create_stuff_documents_chain(model, prompt)

result = chain.invoke({"context": documents})
result

'\n\nApples are red, blueberries are blue, and bananas are yellow.'